# Laboratorio 5. Agentes en el Arcade Learning Environment (ALE): Space Invaders

Fernando Rueda - 23748

CC3092 Deep Learning y Sistemas Inteligentes

En este notebook investigo el Arcade Learning Environment y los espacios de observación y acción de los entornos de Atari, y luego construyo un módulo reutilizable para crear entornos, ejecutar agentes y grabar video de las partidas. No entreno un agente inteligente, el objetivo es dejar funcionando la infraestructura para que un agente simple, aleatorio o de regla fija, interactúe con ALE/SpaceInvaders-v5 y genere un video del juego. Este mismo código será la base para entrenar agentes en el proyecto futuro.

## 1. El Arcade Learning Environment (ALE)

### Qué es ALE y qué problema resuelve

El Arcade Learning Environment es una plataforma que expone los juegos de la consola Atari 2600 como entornos de aprendizaje por refuerzo, con una interfaz común para que un agente reciba observaciones, ejecute acciones y obtenga recompensas. El problema que resuelve es la falta de un banco de pruebas estandarizado y variado para evaluar agentes de propósito general, ALE ofrece decenas de juegos distintos bajo la misma API, de modo que un mismo algoritmo puede medirse en muchas tareas sin cambiar el código. Internamente ALE no reimplementa los juegos, sino que corre las ROM originales de Atari 2600 sobre el emulador Stella, y se limita a leer la pantalla y la memoria del emulador, inyectar las acciones del joystick y traducir el marcador del juego a una señal de recompensa. Así, los juegos con los que interactúa el agente son exactamente los cartuchos originales, y Stella es el motor de emulación que los hace correr.

Fuente: Bellemare, M. et al. (2013). The Arcade Learning Environment: An Evaluation Platform for General Agents, JAIR, y documentación de ale-py, Farama Foundation.

### Variantes de un mismo juego en ALE

Un mismo juego se expone en varias variantes que cambian qué observa el agente y cómo se comporta el entorno. ALE/SpaceInvaders-v5 entrega como observación la imagen RGB de la pantalla, mientras que la variante con sufijo -ram (ALE/SpaceInvaders-ram-v5) entrega los 128 bytes de la memoria de la consola en vez de la imagen. Más allá del tipo de observación, las variantes se ajustan con parámetros. frameskip fija cuántos fotogramas del emulador avanza cada acción, en la familia -v5 el valor por defecto es 4, es decir el agente decide una vez y esa acción se repite cuatro fotogramas. repeat_action_probability es la probabilidad de acciones pegajosas (sticky actions), con la que el entorno repite la acción anterior en lugar de la nueva, en -v5 vale 0.25 e introduce estocasticidad para evitar que el agente memorice una secuencia fija de movimientos. full_action_space decide si el espacio de acción es el conjunto completo de 18 acciones del joystick de Atari o solo el subconjunto mínimo con sentido para ese juego, que en Space Invaders son 6. La versión -v5 es la recomendada por ALE porque estandariza estos valores.

Fuente: Machado, M. et al. (2018). Revisiting the Arcade Learning Environment, JAIR, y documentación de entornos Atari de Gymnasium.

### Frame skipping

El frame skipping consiste en que el agente elige una acción cada k fotogramas y esa misma acción se aplica en los fotogramas intermedios, en vez de decidir en cada uno. Es importante por dos razones. Primero por velocidad, decidir una vez cada cuatro fotogramas reduce a la cuarta parte el número de inferencias del agente y de pasos que hay que procesar, lo que acelera muchísimo la simulación y el entrenamiento sin perder casi información, porque entre fotogramas consecutivos de un juego de Atari el cambio es mínimo. Segundo por la dinámica del aprendizaje, con salto de fotogramas cada acción tiene un efecto más visible sobre el estado siguiente, lo que hace la recompensa menos dispersa y la asignación de crédito temporal más fácil. El costo es de resolución temporal, un salto muy grande puede hacer que el agente pierda eventos rápidos, por eso cuatro es el valor habitual, un compromiso entre velocidad y control fino. Como el salto suele combinarse con tomar el máximo de los dos últimos fotogramas, también corrige el parpadeo de sprites propio del hardware del Atari 2600.

Fuente: Mnih, V. et al. (2015). Human-level control through deep reinforcement learning, Nature, y Machado et al. (2018).

### El juego Space Invaders

En Space Invaders el jugador controla un cañón que se mueve horizontalmente en la parte baja de la pantalla y dispara hacia arriba contra una formación de alienígenas que descienden y disparan de vuelta. El objetivo es destruir a todos los invasores antes de que lleguen abajo, esquivando sus disparos y protegiéndose tras unos búnkeres que se van destruyendo. Se dispone de varias vidas y el episodio termina cuando se pierden todas o los invasores alcanzan al jugador. La puntuación del juego original sube al destruir invasores, valen más los de las filas superiores, y una nave nodriza ocasional da un bono grande. En el entorno de ALE esa puntuación es justamente la señal de recompensa, la recompensa de cada paso es el incremento del marcador del juego en ese paso, de modo que maximizar la recompensa acumulada equivale a maximizar el puntaje. Por eso un agente aleatorio suele obtener una recompensa baja pero no nula, ya que de vez en cuando dispara y acierta a algún invasor por azar.

Fuente: documentación del entorno SpaceInvaders de Gymnasium y ALE.